# Chapter 2.2 — neural ODE examples, explained from scratch

This notebook is a companion to **Section 2.2 of Patrick Kidger's _On Neural Differential Equations_**. It collects every example we discussed: image classification, universal differential equations, Hamiltonian and Lagrangian networks, continuous normalizing flows and trace estimation, latent ODEs, rotational vector fields, and momentum residual networks.

**The one idea to keep in mind:** an ODE tells us how a state changes *right now*. Diffrax takes that rule and computes what the state will be later. A neural network can supply all or part of the rule. Training adjusts its parameters so the resulting trajectory matches data.

| Section | State | What is learned? | What comes out? |
|---|---|---|---|
| 2.2.1 Image classifier | image-like feature map | CNN change rule and final classifier | class probabilities |
| 2.2.2.1 Universal DE | prey and predator counts | correction to known population equations | population trajectory |
| 2.2.2.2 Hamiltonian | position and momentum | energy function | motion |
| 2.2.2.3 Lagrangian | position and velocity | kinetic/potential description | acceleration and motion |
| 2.2.3 Continuous normalizing flow | random point | rule moving a cloud of points | new samples and their densities |
| 2.2.4 Latent ODE | hidden state | starting-state encoder and dynamics | time series at arbitrary times |
| 2.2.5 Residual variants | hidden layer state | structured update rule | stable or reversible layers |

**How to use this:** Run cells top to bottom. Every example uses synthetic data created inside the notebook, so there are no downloads. Training runs are intentionally short; their purpose is to show the mechanism, not to produce benchmark results. On a CPU, the first JAX compilation in each section can take longer than later steps.

**Setup:** Select `/Users/pasha/Desktop/master/master_thesis/sandbox/.venv/bin/python` as the notebook kernel. If using another environment, install `jax`, `diffrax`, `optax`, `matplotlib`, `numpy`, and `ipykernel` first.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import diffrax as dfx
import optax
import numpy as np
import matplotlib.pyplot as plt

print("JAX", jax.__version__, "Diffrax", dfx.__version__)
print("Device:", jax.devices()[0])

def solve_ode(field, y0, t0=0.0, t1=1.0, dt0=0.05, ts=None, solver=None):
    # Integrate dy/dt = field(t, y, args); return saved state(s).
    if solver is None:
        solver = dfx.Tsit5()
    saveat = dfx.SaveAt(t1=True) if ts is None else dfx.SaveAt(ts=ts)
    return dfx.diffeqsolve(
        dfx.ODETerm(field), solver, t0=t0, t1=t1, dt0=dt0,
        y0=y0, saveat=saveat, max_steps=1000,
    ).ys

def fit(params, loss_fn, steps=50, lr=0.02):
    # Fit a PyTree of JAX arrays with Adam and return its loss history.
    optimizer = optax.adam(lr)
    state = optimizer.init(params)

    @jax.jit
    def step(p, s):
        value, grad = jax.value_and_grad(loss_fn)(p)
        updates, s = optimizer.update(grad, s, p)
        return optax.apply_updates(p, updates), s, value

    history = []
    for _ in range(steps):
        params, state, value = step(params, state)
        history.append(float(value))
    return params, np.asarray(history)

def plot_loss(history, title):
    plt.figure(figsize=(5, 2.5))
    plt.plot(history)
    plt.title(title)
    plt.xlabel("training step")
    plt.ylabel("loss")
    plt.grid(alpha=0.3)
    plt.show()

## 1. Image classification: an ODE in place of a stack of layers

**In plain words:** The input image becomes the initial state. A CNN repeatedly says how the image-like feature map should change. Diffrax integrates those tiny changes to depth $T$. A final linear classifier reads the result. Here $t$ means **network depth**, not clock time.

The paper uses 32×32 RGB images and ten classes. For a quick, visible demonstration we use **8×8 RGB pictures and two classes**: horizontal versus vertical stripes. This is the same architecture idea at smaller scale.

**Math-to-code map:** `image_field` is $f_\theta(t,y)$; `solve_ode` computes $y(T)$; `classifier_logits` implements $\ell_\theta(y(T))$; `softmax` turns logits into probabilities.

In [ ]:
image_key = jr.PRNGKey(11)
height = width = 8
base_h = jnp.tile((jnp.arange(height) % 2)[:, None], (1, width))
base_v = jnp.tile((jnp.arange(width) % 2)[None, :], (height, 1))
patterns = jnp.stack([base_h, base_v])
labels = jnp.tile(jnp.arange(2), 12)
noise = 0.12 * jr.normal(image_key, (24, 3, height, width))
images = jnp.clip(patterns[labels, None, :, :] + noise, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(5, 2))
for ax, label, name in zip(axes, [0, 1], ["horizontal", "vertical"]):
    ax.imshow(np.asarray(images[label].transpose(1, 2, 0)))
    ax.set_title(name)
    ax.axis("off")
plt.show()

k1, k2 = jr.split(jr.PRNGKey(12))
image_params = {
    "conv": jr.normal(k1, (3, 3, 3, 3)) * 0.06,
    "head": jr.normal(k2, (2, 3 * height * width)) * 0.02,
    "bias": jnp.zeros(2),
}

def image_field(params, t, y, args):
    # A tiny convolutional network producing an update with the same image shape.
    conv = jax.lax.conv_general_dilated(
        y[None, ...], params["conv"], (1, 1), "SAME",
        dimension_numbers=("NCHW", "OIHW", "NCHW"),
    )[0]
    return 0.5 * jnp.tanh(conv)

def classifier_logits(params, image):
    final = solve_ode(
        lambda t, y, args: image_field(params, t, y, args),
        image, dt0=0.25, solver=dfx.Euler(),
    )[0]
    return params["head"] @ final.reshape(-1) + params["bias"]

def image_loss(params):
    logits = jax.vmap(lambda image: classifier_logits(params, image))(images)
    return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, labels))

print("Initial cross-entropy:", float(image_loss(image_params)))
image_params, image_history = fit(image_params, image_loss, steps=35, lr=0.04)
plot_loss(image_history, "Image classifier training")
logits = jax.vmap(lambda image: classifier_logits(image_params, image))(images)
predictions = jnp.argmax(logits, axis=-1)
print("Training accuracy:", float(jnp.mean(predictions == labels)))
print("First image probabilities:", np.asarray(jax.nn.softmax(logits[0])))

**What to notice:** The CNN never outputs a class. It outputs a *change to the current feature map*. The classifier at the end produces the classes. The ODE state has the same shape throughout this simplified example. A real classifier would normally add more channels and use a conventional residual network; the paper explicitly presents this as an illustration, not a recommended image-classification architecture.

## 2. Universal differential equation: known model + learned correction

Let $x$ be prey and $y$ be predators. The familiar Lotka–Volterra model gives a theory-based rule for their rates of change. A universal differential equation (UDE) keeps that rule and adds a small network that learns what the theory missed:

$$\dot x = \underbrace{\alpha x-\beta xy}_{\text{known rule}}+\underbrace{f_\theta(x,y)}_{\text{learned correction}},\qquad
\dot y = \underbrace{-\gamma y+\delta xy}_{\text{known rule}}+\underbrace{g_\theta(x,y)}_{\text{learned correction}}.$$

**PDF typo:** Equations (2.2) and (2.3) print $-\gamma x$ in the predator equation. The standard Lotka–Volterra death term is $-\gamma y$, which we use here. The correction changes the **rate**, not a single observed population count.

We create observations from a system with an extra effect, then fit a small network while keeping the four theory parameters fixed. That makes the network's job clearly a residual correction.

In [ ]:
lv_times = jnp.linspace(0, 2.0, 21)
lv_initial = jnp.array([1.6, 0.9])

def lv_base(state):
    x, y = state
    return jnp.array([1.0*x - 0.8*x*y, -1.2*y + 0.7*x*y])

def lv_missing_effect(state):
    x, y = state
    return jnp.array([0.12*jnp.sin(y), -0.08*x])

lv_observed = solve_ode(
    lambda t, s, args: lv_base(s) + lv_missing_effect(s),
    lv_initial, t1=2.0, ts=lv_times,
)
lv_uncorrected = solve_ode(
    lambda t, s, args: lv_base(s),
    lv_initial, t1=2.0, ts=lv_times,
)

def init_mlp(key, in_size, hidden_size, out_size, scale=0.1):
    k1, k2 = jr.split(key)
    return {
        "w1": scale * jr.normal(k1, (hidden_size, in_size)),
        "b1": jnp.zeros(hidden_size),
        "w2": scale * jr.normal(k2, (out_size, hidden_size)),
        "b2": jnp.zeros(out_size),
    }

def mlp(params, x):
    return params["w2"] @ jnp.tanh(params["w1"] @ x + params["b1"]) + params["b2"]

lv_params = init_mlp(jr.PRNGKey(21), 2, 12, 2)

def lv_prediction(params):
    def field(t, state, args):
        return lv_base(state) + 0.3 * jnp.tanh(mlp(params, state))
    return solve_ode(field, lv_initial, t1=2.0, ts=lv_times)

def lv_loss(params):
    return jnp.mean((lv_prediction(params) - lv_observed) ** 2)

lv_params, lv_history = fit(lv_params, lv_loss, steps=65, lr=0.04)
lv_fitted = lv_prediction(lv_params)
plot_loss(lv_history, "UDE trajectory error")
plt.figure(figsize=(7, 3))
for component, name in enumerate(["prey", "predators"]):
    plt.plot(lv_times, lv_observed[:, component], label=f"observed {name}")
    plt.plot(lv_times, lv_uncorrected[:, component], "--", label=f"theory only {name}")
    plt.plot(lv_times, lv_fitted[:, component], ":", label=f"UDE {name}")
plt.xlabel("real time")
plt.ylabel("population (scaled)")
plt.legend(ncol=3, fontsize=8)
plt.show()

**What to notice:** The network is inside the differential equation. Diffrax solves the *combined* rule, so an error in today's rate changes all later predictions. We compare whole predicted trajectories against observed trajectories. If we let both the network and $\alpha,\beta,\gamma,\delta$ change freely, the network might take over some behavior that the original parameters were meant to explain.

## 3. Hamiltonian network: learn an energy, derive motion

For position $q$ and momentum $p$, the Hamiltonian $H(q,p)$ is an energy-like function. Hamilton's equations convert its slopes into motion:

$$\dot q=\frac{\partial H}{\partial p},\qquad \dot p=-\frac{\partial H}{\partial q}.$$

**Important:** We do not train the network to predict energy measurements. We train its energy function so that the *motion obtained from its derivatives* matches observed motion. Here we use a small structured energy network for a spring, $H_\theta=a p^2+b q^2$. A larger MLP could also represent $H$; the small version makes every step easy to inspect.

We use known derivatives of synthetic trajectories as training targets. With real measurements, these could be estimated from closely spaced observations or learned by comparing full trajectories.

In [ ]:
spring_times = jnp.linspace(0, 3.0, 31)
spring_start = jnp.array([1.0, 0.2])  # q, p
true_mass, true_k = 1.5, 2.0

def true_hamiltonian_field(t, state, args):
    q, p = state
    return jnp.array([p / true_mass, -true_k * q])

spring_data = solve_ode(
    true_hamiltonian_field, spring_start, t1=3.0, ts=spring_times,
)
ham_params = {"log_a": jnp.array(-0.6), "log_b": jnp.array(-0.8)}

def learned_hamiltonian(params, state):
    q, p = state
    return jnp.exp(params["log_a"]) * p**2 + jnp.exp(params["log_b"]) * q**2

def hamiltonian_field(params, state):
    dH_dq, dH_dp = jax.grad(lambda s: learned_hamiltonian(params, s))(state)
    return jnp.array([dH_dp, -dH_dq])

ham_targets = jax.vmap(lambda s: true_hamiltonian_field(0.0, s, None))(spring_data)

def ham_loss(params):
    predicted_rates = jax.vmap(lambda s: hamiltonian_field(params, s))(spring_data)
    return jnp.mean((predicted_rates - ham_targets) ** 2)

ham_params, ham_history = fit(ham_params, ham_loss, steps=80, lr=0.06)
ham_trajectory = solve_ode(
    lambda t, s, args: hamiltonian_field(ham_params, s),
    spring_start, t1=3.0, ts=spring_times,
)
plot_loss(ham_history, "Hamiltonian motion error")
print("Learned mass:", float(1/(2*jnp.exp(ham_params["log_a"]))))
print("Learned spring stiffness:", float(2*jnp.exp(ham_params["log_b"])))
plt.figure(figsize=(6, 3))
plt.plot(spring_times, spring_data[:, 0], label="observed position")
plt.plot(spring_times, ham_trajectory[:, 0], "--", label="Hamiltonian prediction")
plt.xlabel("time")
plt.ylabel("position")
plt.legend()
plt.show()

**What to notice:** The network produces a *single energy value* for each $(q,p)$. Automatic differentiation obtains its two slopes; Hamilton's equations turn those slopes into a velocity for the state. Diffrax then follows that velocity through time. This built-in physics structure is the inductive bias.

## 4. Lagrangian network: learn a function of position and velocity

The Lagrangian uses $q$ and velocity $v=\dot q$ instead of momentum. For a spring it has the familiar form **kinetic energy minus potential energy**:

$$L_\theta(q,v)=\tfrac12 m_\theta v^2-V_\theta(q).$$

Euler–Lagrange then gives $m\ddot q=-\partial V_\theta/\partial q$. Here we assume the mass is measured and learn the potential $V_\theta$; its derivative gives the force and hence acceleration. This matters because acceleration alone identifies the ratio of stiffness to mass, not their individual values. We use a quadratic base plus a tiny MLP correction, fit acceleration examples, and then use Diffrax to predict an entire trajectory.

In [ ]:
lag_key = jr.PRNGKey(41)
lag_params = {
    "log_k": jnp.array(-0.4),
    "potential_nn": init_mlp(lag_key, 1, 8, 1, scale=0.03),
}
lag_positions = jnp.linspace(-1.5, 1.5, 41)
lag_accelerations = -(true_k / true_mass) * lag_positions

def potential(params, q):
    quadratic = 0.5 * jnp.exp(params["log_k"]) * q**2
    return quadratic + 0.05 * mlp(params["potential_nn"], jnp.array([q]))[0]

def lagrangian(params, q, v):
    return 0.5 * true_mass * v**2 - potential(params, q)

def lagrangian_acceleration(params, q, v):
    # Euler-Lagrange: d/dt(dL/dv) = dL/dq.
    # For L = 0.5*m*v^2 - V(q), this reduces to m*a = -dV/dq.
    dL_dq = jax.grad(lambda position: lagrangian(params, position, v))(q)
    d2L_dv2 = jax.grad(jax.grad(lambda velocity: lagrangian(params, q, velocity)))(v)
    return dL_dq / d2L_dv2

def lag_loss(params):
    fitted = jax.vmap(lambda q: lagrangian_acceleration(params, q, 0.0))(lag_positions)
    return jnp.mean((fitted - lag_accelerations) ** 2)

lag_params, lag_history = fit(lag_params, lag_loss, steps=90, lr=0.04)
lag_trajectory = solve_ode(
    lambda t, s, args: jnp.array([
        s[1], lagrangian_acceleration(lag_params, s[0], s[1])
    ]),
    jnp.array([1.0, 0.2 / true_mass]), t1=3.0, ts=spring_times,
)
plot_loss(lag_history, "Lagrangian acceleration error")
print("Acceleration at q=1:", float(lagrangian_acceleration(lag_params, 1.0, 0.0)))
print("True acceleration at q=1:", -true_k / true_mass)
plt.figure(figsize=(6, 3))
plt.plot(spring_times, spring_data[:, 0], label="observed position")
plt.plot(spring_times, lag_trajectory[:, 0], "--", label="Lagrangian prediction")
plt.xlabel("time")
plt.ylabel("position")
plt.legend()
plt.show()

**What to notice:** The output of the learned Lagrangian is not itself the next position. Its derivatives give acceleration; Diffrax integrates acceleration into velocity and position. The key difference from the Hamiltonian model is **which coordinates you start with**: $(q,p)$ versus $(q,v)$. The paper gives a more general Hessian formula for $L(q,v)$; our structured example makes that Hessian simply the known mass.

## 5. Continuous normalizing flow: reshape a cloud of random points

A normalizing flow starts with a probability distribution we understand, such as a round Gaussian cloud. A learned ODE moves each point; together, the points form a more interesting cloud. The same learned movement can be run **forward to generate samples** and **backward to evaluate real data**.

For clarity, this example learns a two-dimensional *linear neural vector field* $f_\theta(y)=Ay+b$. It can turn a round Gaussian into a shifted, tilted ellipse. The paper's cat-shaped distributions require a nonlinear field, but the logic is identical.

The crucial extra quantity is **log probability density**. If nearby points spread out, density falls; if they squeeze together, density rises. The rate of log-density change is minus the trace of the vector-field Jacobian:

$$\frac{d}{dt}\log p_t(y(t))=-\mathrm{tr}\left(\frac{\partial f_\theta}{\partial y}\right).$$

For our linear field, the Jacobian is exactly $A$, so its trace is cheap to calculate. The next section shows the randomized estimator used when the Jacobian is large.

In [ ]:
cnf_key = jr.PRNGKey(51)
raw = jr.normal(cnf_key, (96, 2))
target_matrix = jnp.array([[1.5, 0.7], [0.25, 0.65]])
target_points = raw @ target_matrix.T + jnp.array([1.4, -0.8])
cnf_params = {"A": jnp.zeros((2, 2)), "b": jnp.zeros(2)}

def cnf_field(params, t, state, args):
    point, log_density_change = state
    velocity = params["A"] @ point + params["b"]
    return velocity, -jnp.trace(params["A"])

def base_log_density(point):
    return -0.5 * (jnp.sum(point**2) + 2*jnp.log(2*jnp.pi))

def cnf_log_density(params, observed_point):
    # Backward solve: data point at t=1 -> base point at t=0.
    # The second state records log p(0) - log p(1).
    # DirectAdjoint gives correct reverse-time gradients in Diffrax 0.7.0 here.
    base_point, backward_change = dfx.diffeqsolve(
        dfx.ODETerm(lambda t, s, args: cnf_field(params, t, s, args)),
        dfx.Tsit5(), t0=1.0, t1=0.0, dt0=-0.1,
        y0=(observed_point, jnp.array(0.0)), max_steps=100,
        adjoint=dfx.DirectAdjoint(),
    ).ys
    return base_log_density(base_point[0]) - backward_change[0]

def cnf_loss(params):
    log_probs = jax.vmap(lambda x: cnf_log_density(params, x))(target_points)
    return -jnp.mean(log_probs)

cnf_params, cnf_history = fit(cnf_params, cnf_loss, steps=85, lr=0.035)

def cnf_sample(params, base_point):
    return solve_ode(
        lambda t, y, args: params["A"] @ y + params["b"],
        base_point, dt0=0.1,
    )[0]

generated_points = jax.vmap(lambda z: cnf_sample(cnf_params, z))(raw)
plot_loss(cnf_history, "CNF negative log likelihood")
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, points, title in zip(
    axes, [raw, target_points, generated_points],
    ["simple base cloud", "target cloud", "learned flow samples"],
):
    ax.scatter(np.asarray(points[:, 0]), np.asarray(points[:, 1]), s=9, alpha=0.6)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()
print("Log density of first target point:", float(cnf_log_density(cnf_params, target_points[0])))

**What to notice:** Training starts with a real target point, runs the ODE backward, and asks how probable its starting point was under the known Gaussian. The density correction accounts for stretching/squeezing along the way. Sampling runs the same learned ODE forward. This is a *generative* model; no class labels are needed.

The paper's Figure 2.2 makes a **two-dimensional distribution** look like a cat silhouette. That is different from image generation, where each point in the distribution is itself an entire image.

### 5a. Trace–Jacobian estimation: avoid building a huge Jacobian

In two dimensions, calculating the trace exactly is easy. For an image with thousands of dimensions it is expensive to inspect each diagonal entry. Hutchinson's identity gives an unbiased estimate using random vectors $\varepsilon$ with independent entries $+1$ or $-1$:

$$\mathrm{tr}(J)=\mathbb E_\varepsilon[\varepsilon^\top J\varepsilon].$$

In code, `jax.jvp` computes $J\varepsilon$ **without constructing the full Jacobian**. One random estimate is noisy; averaging several improves it. Here we use a 32-dimensional linear field so we know the exact answer and can check the estimate.

In [ ]:
trace_key, eps_key = jr.split(jr.PRNGKey(52))
large_A = 0.08 * jr.normal(trace_key, (32, 32)) + 0.4*jnp.eye(32)
exact_trace = jnp.trace(large_A)
probe_vectors = jr.rademacher(eps_key, (256, 32)).astype(jnp.float32)

def one_trace_estimate(epsilon):
    _, jacobian_times_epsilon = jax.jvp(
        lambda x: large_A @ x, (jnp.zeros(32),), (epsilon,)
    )
    return epsilon @ jacobian_times_epsilon

estimates = jax.vmap(one_trace_estimate)(probe_vectors)
print("Exact trace:", float(exact_trace))
print("One random estimate:", float(estimates[0]))
print("Mean of 256 estimates:", float(jnp.mean(estimates)))
plt.figure(figsize=(6, 2.5))
plt.plot(np.asarray(jnp.cumsum(estimates)/jnp.arange(1, 257)))
plt.axhline(float(exact_trace), color="black", linestyle="--", label="exact trace")
plt.xlabel("number of random probes")
plt.ylabel("running estimate")
plt.legend()
plt.show()

## 6. Latent ODE: a hidden state evolving in real time

A latent ODE describes a time series using a hidden state $y(t)$ that we do **not** observe directly. An **encoder** reads measured values and times and guesses a distribution over the hidden starting state. The hidden state follows a neural ODE. A **decoder** turns the hidden state at any requested time into an observation.

This is a small version of the paper's VAE design:

1. Encoder: irregular observations $\rightarrow$ mean and uncertainty for the starting state.
2. Sample a starting state; Diffrax evolves it in **actual time**.
3. Decoder: hidden state $\rightarrow$ predicted observation at each measurement time.
4. Train with reconstruction error plus a KL term that keeps the starting-state distribution near a standard Gaussian.

We use short decaying oscillations measured at different, uneven time grids. The hidden dynamics is a learned two-dimensional linear neural field. The decoder reads its first coordinate. A larger model could learn a nonlinear field and decoder. Training is deliberately small; the plot is a mechanism check, not a forecasting benchmark.

In [ ]:
latent_key = jr.PRNGKey(61)
time_key, initial_key, encoder_key = jr.split(latent_key, 3)
irregular_times = jnp.sort(jr.uniform(time_key, (8, 5), minval=0.05, maxval=3.0), axis=1)
true_initials = jr.normal(initial_key, (8, 2)) * 0.7
true_A = jnp.array([[-0.2, 1.8], [-1.8, -0.2]])

def observed_series(y0, times):
    trajectory = solve_ode(
        lambda t, y, args: true_A @ y,
        y0, t1=3.0, ts=times,
    )
    return trajectory[:, 0]

series_data = jax.vmap(observed_series)(true_initials, irregular_times)
latent_params = {
    "encoder": init_mlp(encoder_key, 10, 16, 4, scale=0.08),
    "A": jnp.array([[-0.15, 1.6], [-1.6, -0.15]]),
}

def latent_encode(params, times, values):
    stats = mlp(params["encoder"], jnp.concatenate([times, values]))
    return stats[:2], jnp.clip(stats[2:], -3.0, 1.0)

def latent_decode(params, z0, times, end_time=3.0):
    hidden = solve_ode(
        lambda t, y, args: params["A"] @ y,
        z0, t1=end_time, ts=times,
    )
    return hidden[:, 0]

def latent_loss(params, epsilons):
    def one_series(times, values, epsilon):
        mean, logvar = latent_encode(params, times, values)
        z0 = mean + jnp.exp(0.5*logvar)*epsilon
        reconstruction = latent_decode(params, z0, times)
        recon_loss = jnp.mean((reconstruction-values)**2)
        kl = 0.5*jnp.sum(jnp.exp(logvar)+mean**2-1-logvar)
        return recon_loss + 0.01*kl
    return jnp.mean(jax.vmap(one_series)(irregular_times, series_data, epsilons))

latent_optimizer = optax.adam(0.025)
latent_opt_state = latent_optimizer.init(latent_params)

@jax.jit
def latent_step(params, opt_state, key):
    eps = jr.normal(key, (8, 2))
    value, grad = jax.value_and_grad(latent_loss)(params, eps)
    updates, opt_state = latent_optimizer.update(grad, opt_state, params)
    return optax.apply_updates(params, updates), opt_state, value

latent_history = []
for step_number in range(85):
    latent_key, step_key = jr.split(latent_key)
    latent_params, latent_opt_state, value = latent_step(
        latent_params, latent_opt_state, step_key
    )
    latent_history.append(float(value))

plot_loss(latent_history, "Latent ODE VAE objective")
mean0, logvar0 = latent_encode(latent_params, irregular_times[0], series_data[0])
dense_times = jnp.linspace(0.05, 5.0, 100)
forecast = latent_decode(latent_params, mean0, dense_times, end_time=5.0)
plt.figure(figsize=(7, 3))
plt.scatter(irregular_times[0], series_data[0], label="irregular observations")
plt.plot(dense_times, forecast, label="decoded latent trajectory")
plt.axvline(3.0, color="gray", linestyle="--", label="end of observed interval")
plt.xlabel("real time")
plt.ylabel("observed value")
plt.legend()
plt.show()
print("Encoded starting-state mean:", np.asarray(mean0))
print("The part after t=3 is extrapolation; its accuracy is not guaranteed.")

**What to notice:** Unlike the image classifier, $t$ here is real observation time. Diffrax can save a prediction at each series' own timestamps, even if they are uneven or differ across examples. The hidden state can have more dimensions than each observation. We can also ask for values between measurements, or later than the last one—though extrapolation quality must be checked rather than assumed.

## 7. Rotational vector fields: updates that circulate

Ordinary residual networks add a new update at every layer. If many updates keep stretching or shrinking in the same direction, signals and gradients can explode or vanish. A rotational vector field arranges two state components so one pushes the other and vice versa. Its continuous-time motion circles around rather than systematically expanding or contracting.

For a two-dimensional toy state $(y,z)$, consider $\dot y=\omega z$, $\dot z=-\omega y$. The squared radius $y^2+z^2$ stays constant. We solve that ODE with Diffrax. We also compare two discrete updates: naive forward Euler and the paper's rotation-like **semi-implicit Euler** step, which uses the new $z$ when updating $y$.

In [ ]:
omega = 2.0
rotation_times = jnp.linspace(0, 12.0, 121)
rotation_start = jnp.array([1.0, 0.0])

def rotation_field(t, state, args):
    y, z = state
    return jnp.array([omega*z, -omega*y])

continuous_rotation = solve_ode(
    rotation_field, rotation_start, t1=12.0, ts=rotation_times,
)

step_size = 0.1
def euler_step(state):
    return state + step_size * rotation_field(0.0, state, None)

def semi_implicit_step(state):
    y, z = state
    new_z = z - step_size*omega*y
    new_y = y + step_size*omega*new_z
    return jnp.array([new_y, new_z])

def discrete_path(step_fn, count=120):
    states = [rotation_start]
    for _ in range(count):
        states.append(step_fn(states[-1]))
    return jnp.stack(states)

naive_path = discrete_path(euler_step)
structured_path = discrete_path(semi_implicit_step)
plt.figure(figsize=(6, 3))
for states, name in [
    (continuous_rotation, "Diffrax continuous rotation"),
    (naive_path, "naive Euler"),
    (structured_path, "semi-implicit update"),
]:
    plt.plot(rotation_times, jnp.linalg.norm(states, axis=1), label=name)
plt.xlabel("time / layer depth")
plt.ylabel("state size")
plt.legend(fontsize=8)
plt.show()

**What to notice:** The underlying rotation does not inherently stretch the state. Yet a poor *discrete* approximation can still grow, as naive Euler does here. The paper's structured residual update is closer to the desired rotation. This toy uses the identity activation and one scalar coupling; the paper describes a higher-dimensional trainable $K$ and activation $\sigma$.

## 8. Momentum residual networks: remember the previous update

A momentum residual network keeps two things at each layer: its state $y$ and an update direction $v$:

$$v_{j+1}=\gamma v_j+(1-\gamma)f_\theta(y_j),\qquad y_{j+1}=y_j+v_{j+1}.$$

With $\gamma=0.9$, the new direction is 90% old direction and 10% newly proposed direction. The auxiliary $v$ is an update memory; it need not be physical velocity.

The paper's important observation is **reversibility**. Given $(y_{j+1},v_{j+1})$, we can recover the previous state exactly in arithmetic. During backpropagation this can reduce memory use: earlier activations can be recomputed rather than stored. We verify the reverse formula numerically. Diffrax also solves the related continuous second-order equation, showing the ODE connection.

In [ ]:
gamma = 0.9

def proposed_update(y):
    return -0.25*y + 0.15*jnp.sin(y)

def momentum_forward(y, v):
    next_v = gamma*v + (1-gamma)*proposed_update(y)
    next_y = y + next_v
    return next_y, next_v

def momentum_reverse(next_y, next_v):
    old_y = next_y - next_v
    old_v = (next_v - (1-gamma)*proposed_update(old_y))/gamma
    return old_y, old_v

y0, v0 = jnp.array(1.0), jnp.array(0.0)
momentum_states = [(y0, v0)]
for _ in range(35):
    momentum_states.append(momentum_forward(*momentum_states[-1]))

recovered = momentum_states[-1]
for _ in range(35):
    recovered = momentum_reverse(*recovered)

print("Original (y, v):", float(y0), float(v0))
print("Recovered after 35 reverse steps:", float(recovered[0]), float(recovered[1]))

epsilon = 1/(1-gamma)
momentum_times = jnp.arange(36, dtype=jnp.float32)
continuous_momentum = solve_ode(
    lambda t, s, args: jnp.array([s[1], (proposed_update(s[0])-s[1])/epsilon]),
    jnp.array([y0, v0]), t1=35.0, dt0=0.1, ts=momentum_times,
)
plt.figure(figsize=(6, 3))
plt.plot(momentum_times, jnp.array([state[0] for state in momentum_states]),
         label="momentum residual layers")
plt.plot(momentum_times, continuous_momentum[:, 0], "--",
         label="related continuous ODE")
plt.xlabel("layer / continuous time")
plt.ylabel("state y")
plt.legend()
plt.show()

**What to notice:** The discrete and continuous curves need not match exactly at such a large step size. The conceptual connection is that both keep a state and an update direction. Reversibility is an algebraic property of the discrete recurrence. Floating-point reversal can accumulate small errors over many layers, so numerical equality is approximate.

## One-page recap

- **Image classifier:** a CNN says how features should change as network depth progresses; a final layer reads off classes.
- **Universal DE:** a trusted scientific equation remains in place; a network corrects its missing rate terms.
- **Hamiltonian:** learn an energy $H(q,p)$; its derivatives determine motion.
- **Lagrangian:** learn $L(q,v)$; Euler–Lagrange determines acceleration and motion.
- **Continuous normalizing flow:** move simple random samples continuously into a target distribution; track density as space stretches or squeezes.
- **Trace estimation:** random probe vectors estimate the density-change calculation without forming a huge Jacobian.
- **Latent ODE:** infer a hidden starting state from irregular observations, evolve it in real time, decode measurements when needed.
- **Rotational field:** couple state components so the continuous dynamics circulate rather than systematically expand or contract.
- **Momentum residual network:** carry the previous update direction; the discrete update can be reversed to save memory during training.

**Questions to ask whenever you meet a new neural DE:** What is the state? Is $t$ real time or network depth? Which part is fixed by theory? Which part is learned? What observations define the loss? What does the solver return?

**Sources:** Patrick Kidger, _On Neural Differential Equations_, Chapter 2.2 and Figures 2.2–2.4 (local PDF: `/Users/pasha/.codex/.chatgpt-projects/g-p-6aaa8c5b21348191b7840411b351deda/sources/ordinary neural equations.pdf`). The notebook uses the [Diffrax `diffeqsolve` API](https://docs.kidger.site/diffrax/api/diffeqsolve/), [SaveAt](https://docs.kidger.site/diffrax/api/saveat/), and the [official Neural ODE example](https://docs.kidger.site/diffrax/examples/neural_ode/). The toy datasets and smaller models here are teaching adaptations, not numerical reproductions of the paper's experiments.